In [1]:
import os
import glob
import re
from collections import defaultdict

import pandas as pd
import originpro as op
import OriginExt as oe


target_folder = r"C:\Users\이승윤\Desktop\LSV"

REF_DICT = {
    "Hg/HgO": 0.098,
    "Ag/AgCl (3.5M KCl)": 0.205,
}
CURRENT_REF = "Hg/HgO"
PH_VALUE = 14
ELECTRODE_AREA = 0.5  # cm2
RHE_SHIFT = REF_DICT[CURRENT_REF] + 0.059 * PH_VALUE

save_path = os.path.join(target_folder, "LSV_Origin_Modified.opju")


def connect_origin():
    """Attach to an existing Origin instance, or start one if needed."""
    import originpro.config as opc

    try:
        op.attach()
    except Exception as attach_error:
        message = str(attach_error).lower()
        is_rpc_error = "rpc" in message or "원격 프로시저" in message

        if is_rpc_error and opc.oext and "_app" in vars(opc.po):
            opc.po._app = None

        try:
            op.set_show(False)
        except Exception as start_error:
            raise RuntimeError(
                "Origin에 연결할 수 없습니다. Origin을 실행한 뒤 다시 시도하세요."
            ) from start_error

    print("Origin 연결 완료")


def force_xy_designations(wks, pattern):
    """Set and verify the requested repeating column designation pattern."""
    ncols = wks.cols
    if ncols <= 0 or ncols % len(pattern) != 0:
        raise RuntimeError(
            f"열 수와 패턴 길이가 맞지 않습니다: 열={ncols}, 패턴={pattern}"
        )

    expected = pattern * (ncols // len(pattern))
    codes = {"X": oe.COLTYPE_X, "Y": oe.COLTYPE_Y}

    for index, axis in enumerate(expected):
        wks.obj.Columns(index).SetType(codes[axis])

    actual = [wks.obj.Columns(index).GetType() for index in range(ncols)]
    wanted = [codes[axis] for axis in expected]
    if actual != wanted:
        raise RuntimeError(
            f"열 속성 검증 실패: {wks.lt_range()}\n"
            f"기대: {wanted}\n실제: {actual}"
        )

    print(f"  ✅ {wks.lt_range()}: {expected} 지정 및 검증 완료")


def create_sheet(df, sheet_name, pattern):
    """Create a worksheet, populate labels/data and set its X/Y pattern."""
    wks = op.new_sheet("w", sheet_name)
    if wks is None:
        raise RuntimeError(f"워크시트 생성 실패: {sheet_name}")

    wks.name = sheet_name
    wks.from_df(df)
    for index, labels in enumerate(df.columns):
        wks.set_label(index, str(labels[0]), "L")
        wks.set_label(index, str(labels[1]), "U")
        wks.set_label(index, str(labels[2]), "C")

    force_xy_designations(wks, pattern)
    return wks


def create_lsv_graphs(wks, base_name):
    """(새로 추가됨) Create uncompensated and compensated LSV line graphs."""
    # 1. 비보상(Uncompensated) 그래프 생성
    gp_uncomp = op.new_graph(template='Line')
    gp_uncomp.lname = f"{base_name}_Uncomp_LSV"
    gl_uncomp = gp_uncomp[0]
    
    # 2. 보상(Compensated) 그래프 생성
    gp_comp = op.new_graph(template='Line')
    gp_comp.lname = f"{base_name}_Comp_LSV"
    gl_comp = gp_comp[0]
    
    # 3. 스캔 횟수만큼 반복하며 데이터 선(Plot) 추가 (7칸씩 점프)
    for i in range(0, wks.cols, 7):
        # [비보상 그래프] X = V vs. RHE (인덱스 i+1), Y = Current Density (인덱스 i+3)
        gl_uncomp.add_plot(wks, i+3, i+1)
        
        # [보상 그래프] X = V vs. RHE iR-free (인덱스 i+5), Y = Comp Current Density (인덱스 i+6)
        gl_comp.add_plot(wks, i+6, i+5)
        
    # 4. 마무리 (축 자동 조절 및 다중 스캔 시 색상 자동 분리)
    gl_uncomp.rescale()
    gl_uncomp.group()
    
    gl_comp.rescale()
    gl_comp.group()
    
    print(f"  ✅ {base_name} LSV 그래프 2종(비보상, 보상) 자동 생성 완료")


def read_nyquist(file, original_fname):
    """Read Real Z from column B and -Imaginary Z from column C."""
    raw = pd.read_csv(
        file,
        sep=r"\s+",
        skiprows=63,
        header=None,
        encoding="latin1",
    )
    if raw.shape[1] < 3:
        raise ValueError(
            f"Nyquist 데이터에 최소 3열이 필요합니다. 현재: {raw.shape[1]}열"
        )

    real_z = pd.to_numeric(raw.iloc[:, 1], errors="coerce")
    minus_imag_z = pd.to_numeric(raw.iloc[:, 2], errors="coerce")
    valid = real_z.notna() & minus_imag_z.notna()
    real_z = real_z[valid].reset_index(drop=True)
    minus_imag_z = minus_imag_z[valid].reset_index(drop=True)
    if real_z.empty:
        raise ValueError("유효한 Real Z / -Imaginary Z 데이터가 없습니다.")

    df = pd.concat([real_z, minus_imag_z], axis=1)
    df.columns = pd.MultiIndex.from_tuples([
        ("Real Z", "Ohm", original_fname),
        ("-Imaginary Z", "Ohm", original_fname),
    ])
    return df


def extract_rs(df_nyquist):
    """Use Real Z at the first minimum -Imaginary Z row as Rs."""
    min_position = int(df_nyquist.iloc[:, 1].to_numpy().argmin())
    return float(df_nyquist.iloc[min_position, 0])


def read_lsv(file, original_fname, rs):
    """Read LSV potential/current data and calculate RHE/iR-free columns."""
    raw = pd.read_csv(
        file,
        sep=r"\s+",
        skiprows=62,
        header=None,
        encoding="latin1",
    )
    if raw.shape[1] < 8:
        raise ValueError(
            f"LSV 데이터에 최소 8열이 필요합니다. 현재: {raw.shape[1]}열"
        )

    potential = pd.to_numeric(raw.iloc[:, 6], errors="coerce")
    current = pd.to_numeric(raw.iloc[:, 7], errors="coerce")
    valid = potential.notna() & current.notna()
    potential = potential[valid].reset_index(drop=True)
    current = current[valid].reset_index(drop=True)
    if potential.empty:
        raise ValueError("유효한 전위/전류 데이터가 없습니다.")

    potential_rhe = potential + RHE_SHIFT
    current_density = current / ELECTRODE_AREA
    current_amp = current * 0.001
    potential_comp_rhe = potential - current_amp * rs + RHE_SHIFT
    rs_column = pd.Series([rs] + [float("nan")] * (len(potential) - 1))

    df = pd.concat(
        [
            potential,
            potential_rhe,
            current,
            current_density,
            rs_column,
            potential_comp_rhe,
            current_density,
        ],
        axis=1,
    )
    df.columns = pd.MultiIndex.from_tuples([
        (f"Potential vs. {CURRENT_REF}", "V", original_fname),
        ("Potential vs. RHE", "V", original_fname),
        ("Current", "mA", original_fname),
        ("Current Density", "mA/cm2", original_fname),
        ("Series Resistance (Rs)", "Ohm", f"{original_fname} Rs"),
        (
            "Potential vs. RHE (iR-free)",
            "V",
            f"{original_fname} (Comp)",
        ),
        (
            "Current Density",
            "mA/cm2",
            f"{original_fname} (Comp)",
        ),
    ])
    return df


def process_mpt_final():
    if not os.path.isdir(target_folder):
        raise FileNotFoundError(f"폴더가 없습니다: {target_folder}")
    if ELECTRODE_AREA <= 0:
        raise ValueError("전극 면적은 0보다 커야 합니다.")

    mpt_files = sorted(glob.glob(os.path.join(target_folder, "*.mpt")))
    if not mpt_files:
        print("mpt 파일을 찾을 수 없습니다.")
        return

    lsv_groups = defaultdict(list)
    nyquist_files = []
    for file in mpt_files:
        fname = os.path.splitext(os.path.basename(file))[0]
        nyquist_match = re.fullmatch(r"(.+)_Nyquist", fname, flags=re.IGNORECASE)
        if nyquist_match:
            nyquist_files.append((nyquist_match.group(1), file, fname))
            continue

        scan_match = re.fullmatch(r"(.+)_(\d+)", fname)
        if scan_match:
            base_name = scan_match.group(1)
            scan_num = int(scan_match.group(2))
        else:
            base_name = fname
            scan_num = 0
        lsv_groups[base_name].append((scan_num, file, fname))

    rs_by_group = {}
    nyquist_data = []
    for base_name, file, fname in nyquist_files:
        group_key = base_name.casefold()
        if group_key in rs_by_group:
            raise ValueError(f"같은 그룹의 Nyquist가 중복됩니다: {base_name}")
        try:
            df_nyquist = read_nyquist(file, fname)
            rs = extract_rs(df_nyquist)
        except Exception as e:
            raise RuntimeError(f"Nyquist의 Rs 계산 실패: {fname}") from e

        rs_by_group[group_key] = rs
        nyquist_data.append((base_name, fname, df_nyquist))
        print(f"✅ {fname}: Rs = {rs:.6g} Ω")

    connect_origin()
    try:
        op.new()
        default_book = op.find_book()
        if default_book:
            default_book.destroy()

        created_sheets = 0
        failed_files = []

        for base_name, files_info in lsv_groups.items():
            sheet_name = f"LSV_{base_name}"
            group_key = base_name.casefold()
            rs = rs_by_group.get(group_key, 0.0)
            print(f"\n➡ 처리 중: {sheet_name}; Rs = {rs:.6g} Ω")

            scans = []
            for scan_num, file, fname in sorted(files_info, key=lambda item: item[0]):
                try:
                    df_scan = read_lsv(file, fname, rs)
                    scans.append(df_scan)
                    print(f"  읽기 완료: {fname} ({len(df_scan)}행)")
                except Exception as e:
                    failed_files.append(fname)
                    print(f"  [읽기 실패] {fname}: {e}")

            if not scans:
                print("  [건너뜀] 유효한 스캔이 없습니다.")
                continue

            # (수정됨) 생성된 워크시트 객체를 wks 변수로 받음
            wks = create_sheet(
                pd.concat(scans, axis=1),
                sheet_name,
                pattern="XXYYYXY",
            )
            
            # (새로 추가됨) 시트를 만든 직후 바로 그래프 2종을 생성
            create_lsv_graphs(wks, base_name)
            
            created_sheets += 1

        for base_name, fname, df_nyquist in nyquist_data:
            sheet_name = f"Nyquist_{base_name}"
            print(f"\n➡ 처리 중: {sheet_name}")
            create_sheet(df_nyquist, sheet_name, pattern="XY")
            created_sheets += 1

        if created_sheets == 0:
            print("\n저장할 데이터가 없습니다.")
            return

        op.save(save_path)
        print(f"\n✅ {created_sheets}개 시트 저장 완료!\n파일: {save_path}")
        if failed_files:
            print("\n읽기 실패로 제외된 파일:\n" + "\n".join(failed_files))
    finally:
        try:
            op.exit()
            print("Origin 종료 완료")
        except Exception as close_error:
            print(f"Origin 종료 요청 실패: {close_error}")


if __name__ == "__main__":
    process_mpt_final()

✅ Pt_Nyquist: Rs = 1.86651 Ω
Origin 연결 완료

➡ 처리 중: LSV_Pd; Rs = 0 Ω
  읽기 완료: Pd_1 (229행)
  읽기 완료: Pd_2 (229행)


: 